# Freshwater flux from the Greenland ice sheet - a tool for hydropower planning


This notebook is a demonstrator from the **Digital Twin Component for Ice Sheets
(DTC-IS)**, an ESA-funded, EO-driven digital twin project.

DTC-IS integrates Earth Observation, in-situ data, reanalysis products, and
advanced data-driven models to provide a dynamic and science-based
representation of the Greenland and Antarctic ice sheets, including their ice
shelves. It supports the monitoring, understanding, and prediction of ice-sheet
change, delivering actionable information for research, policy, and
scenario-based exploration.

[Project Website](https://dtc-ice-sheets.org/) ·
[Interactive Dashboard](https://dashboards.dtc-ice-sheets.org/) ·
[Query API](https://query.dtc-ice-sheets.org/docs)

---


Hydropower is at the heart of Greenland's energy system, providing most of its
electricity and supporting the country's renewable energy ambitions. Greenland's
many natural lakes act as dependable reservoirs, refilled each year by meltwater
from the ice sheet. Climate projections point to more meltwater reaching these reservoirs over the
coming decades, which is an opportunity for clean energy growth and for future
power exports. What the twenty-year record here shows most clearly is how
different one season is from the next, which is the variability a planner has to
design around.

This notebook predicts **weekly proglacial discharge** through a melt season,
and from that the two numbers a hydropower operator plans around: **how much
water arrives in total**, and **how large the peak gets**. It then runs the same
model forward under future climate scenarios.

## How it works, in one paragraph

Each melt season is described by a single number - how much meltwater the
climate model says ran off the catchment that year. The **GRF module** puts that
model output onto a common mesh of the catchment first, so that sources at
different resolutions can be used interchangeably. Then, for each week of the
melt season *separately*, a relationship is fitted between that seasonal number
and the discharge measured in that week, across twenty years of record. Feeding
one seasonal number through all twenty-two weekly relationships at once returns
a complete season: a shape, a total, and a peak.

## What to do

Run the cells in order. Cells marked **USER INPUT** are the ones to change -
everything else follows from them. Every dataset and every fit comes from the
[Query API](https://query.dtc-ice-sheets.org), so a token is needed; the next
cells ask for one. The working parts live in `api_helpers.py`,
`uc1_hydro_helpers.py` and `uc1_plotting_helpers.py` under
`dtc_is_notebook_helpers`, so a step can be changed in one place without
disturbing the others.

---


## Notebook preparations

In [ ]:
#@title Install dependencies and fetch the inputs {display-mode: "form"}
import sys

if "google.colab" in sys.modules:
    print("Installing dependencies...")
    %pip install -q xarray zarr fsspec aiohttp ipywidgets==7.7.1 dtc-query-client jupyter-ui-poll
    # Ensure the newest version of the helpers package is installed
    %pip uninstall -y dtc_is_notebooks
    %pip install -q --upgrade --force-reinstall --no-deps --no-cache-dir \
        git+https://github.com/DTC-Ice-Sheets/dtc_is_notebooks.git
    print("Dependencies installed.")

In [ ]:
#@title Import the libraries {display-mode: "form"}
import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr
from IPython.display import display as ipydisplay

from dtc_is_notebook_helpers import api_helpers as api
from dtc_is_notebook_helpers import uc1_hydro_helpers as uc1
from dtc_is_notebook_helpers import uc1_plotting_helpers as uc1_plot
from dtc_is_notebook_helpers.authentication_widget import request_api_token, running_in_colab
from dtc_is_notebook_helpers.cell_spinner import enable_cell_spinner, set_message

# Show an animation at the bottom of each cell while it is running
if not running_in_colab():
    enable_cell_spinner()

### Enter DTC-IS API token

Please run the cell below, paste your DTC-IS API token into the box, and press 'Submit'. If you don’t have a token yet, you’ll need to generate one first using the link next to the box. A quick test query will check the token, and once it passes you can move on to the next cell! To submit a different token you will need to run the cell again.

---

In [ ]:
#@title DTC-IS API token {display-mode: "form"}
token_widget = request_api_token()
await token_widget.wait_until_authenticated()
client = token_widget.get_client()

### Setup

**USER INPUT** - the settings below control everything that follows.

- **Time step** - the length of a time step. `7` for weekly (recommended), `14`
  for fortnightly, `1` for daily. Shorter steps resolve more but are noisier;
  the skill printed in Step 4 will tell you the cost.
- **Forcing dataset** - which runoff dataset drives the prediction. Three are available:
  `RACMO55`, the 5.5 km run covering 2006–2025; `RACMO`, the same model at
  1 km, sampled at 2 km for this chain, whose last complete melt season is
  2020; and `MAR`, a different climate model at
  5 km, also to 2025. The three are compared directly in Step 7.
- **Predictor** - the single number each melt season is described by.
  `catchment sum` is the total of the fitted runoff fields, and is the default.
  The other two are the leading FTS coefficient of the runoff fields and of the
  lake fields; either is only offered if that FTS step returned something
  usable. Step 3 compares all three against the measured seasonal volume before
  you commit to one. Choosing the lake coefficient restricts everything that
  follows to the eight seasons the lake record covers, and the harmonisation
  term in Step 9 is measured on catchment runoff, so it applies only to
  `catchment sum`.


In [ ]:
#@title Setup - choose the forcing and time step {display-mode: "form"}
DATASET = {                        # the published collection each forcing is fitted from
    "RACMO":   "UC1_WATSON_RUNOFF_INPUT_RACMO",
    "MAR":     "UC1_WATSON_RUNOFF_INPUT_MAR",
    "RACMO55": "UC1_WATSON_RUNOFF_INPUT_RACMO55",
}
LAKE_DATASET = "UC1_WATSON_LAKE_INTENSITY"           # supraglacial lake cover, 2014-2021
START_DATE, END_DATE = "2006-01-01", "2026-01-01"    # the whole record
LABELS = {                         # how each forcing is named in tables and plots
    "RACMO":   "RACMO 2 km",
    "RACMO55": "RACMO 5.5 km",
    "MAR":     "MAR 5 km",
}

# House style: measured is always blue, modelled always orange.
OBS, MODEL = uc1_plot.MEASURED, uc1_plot.MODELLED
plt.rcParams.update({"figure.figsize": (11, 4.2), "font.size": 11,
                     "axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "grid.color": "#e2e2e2",
                     "figure.facecolor": "white", "axes.facecolor": "white"})

forcing_dropdown = widgets.Dropdown(
    options=[(f"{LABELS[key]} ({key})", key) for key in ("RACMO55", "RACMO", "MAR")],
    value="RACMO55",
    description="Forcing dataset:",
    style={"description_width": "initial"},
)
predictor_dropdown = widgets.Dropdown(
    options=["catchment sum", "FTS factor 1", "lake FTS factor 1"],
    value="catchment sum",
    description="Predictor:",
    style={"description_width": "initial"},
)
step_days_dropdown = widgets.Dropdown(
    options=[("weekly (7 days)", 7), ("fortnightly (14 days)", 14), ("daily (1 day)", 1)],
    value=7,
    description="Time step:",
    style={"description_width": "initial"},
)
ipydisplay(widgets.VBox([forcing_dropdown, predictor_dropdown, step_days_dropdown]))

## Step 1 - GRF: harmonise onto the mesh

The **Gaussian random field module** fits every snapshot onto a fixed 295-node
mesh of the catchment. This is the module that makes the rest possible: whatever
goes in - a climate model at 5 km, another at 2 km, satellite data at 20 km -
comes out on the same nodes and can be compared, swapped or combined.

The module reads the published collection month by month, one value for every
grid cell along with where that cell sits, and fits each month on its own.

Points falling outside the mesh are dropped first. INLA refuses a fit outright
when any observation lies outside it, and the snapshots are cut to a square box
while the mesh is built around the catchment, so there are always some.

The mesh is deliberately wider than the catchment. A field fitted right up to
the edge of its own domain is distorted there, so the mesh is given a margin,
and a good share of its nodes sit outside the basin - some by more than twenty
kilometres. They belong to the fit. They are left out of anything this notebook
calls a catchment mean, and the maps are clipped to the outline.

Each fit is one to three seconds, so a twenty-year run takes a few minutes on
the server. The API caches a run against its inputs, so asking for the same
window again returns in seconds.


In [ ]:
#@title Run the GRF module {display-mode: "form"}
FORCING = forcing_dropdown.value
STEP_DAYS = step_days_dropdown.value
print(f"forcing    {LABELS[FORCING]}")
print(f"time step  {STEP_DAYS} days\n")

# The catchment outline and the module's own mesh, which ship with the helpers
# package. The outline masks the catchment means and clips the maps, and the
# mesh is drawn over the maps to show where the fit is actually evaluated.
basin = uc1.load_basin()
mesh_vertices, mesh_triangles = uc1.load_mesh()

# The deployed module returns its field interpolated onto a regular grid, so a
# column here is a grid cell rather than a mesh node. Cells outside the mesh
# have no value, and are dropped along with those outside the catchment.
grf_geozarr_url = await api.run_grf_job(client, DATASET[FORCING], START_DATE, END_DATE,
                                        progress=set_message)
grf_ds = xr.open_dataset(grf_geozarr_url, engine="zarr")
grf = uc1.grf_fields(grf_ds)
fields, points = grf["fields"], grf["points"]
in_basin = uc1.basin_nodes(points, basin) & np.isfinite(fields.to_numpy()).all(axis=0)
monthly_runoff = uc1.catchment_mean(fields, in_basin)
monthly_runoff = monthly_runoff[monthly_runoff.index.month.isin(uc1.MELT_MONTHS)]
print(f"{fields.shape[1]} grid cells, {in_basin.sum()} of them inside the "
      f"catchment outline with a value at every step")
print(f"{fields.shape[0]} fields x {fields.shape[1]} columns, "
      f"{fields.index.min():%b %Y} to {fields.index.max():%b %Y}")

uc1_plot.plot_runoff_map(fields, points, basin, mesh_vertices, mesh_triangles, FORCING)
plt.show()


## Step 2 - FTS: compress each season

The **functional time series module** treats each mesh node's melt season as a
curve rather than five separate numbers, finds the patterns those curves share,
and returns a few coefficients per season.

Those coefficients are weights on shared patterns, not physical quantities. A
season with a large first coefficient resembles the dominant pattern across the
record more strongly than one with a small coefficient, and the sign only says
which way round. They are the compressed form the next step uses, and are shown
here in full rather than plotted, because a plot of them invites comparisons
between seasons that the numbers do not support.



In [ ]:
#@title Run the FTS module {display-mode: "form"}
coefficients = None

try:
    fts_geozarr_url = await api.run_fts_job(client, grf_geozarr_url, curve_time_points=5,
                                            num_pca=2, num_factors=3, progress=set_message)
    fts_ds = xr.open_dataset(fts_geozarr_url, engine="zarr")
    fts = uc1.fts_coefficients(fts_ds)
    coefficients = fts["coefficients"]
    coefficients.index = sorted({t.year for t in fields.index})[:len(coefficients)]
    print(fts["layout"])
    print(f"{coefficients.shape[1]} coefficients describe each of the "
          f"{len(coefficients)} melt seasons")
    uc1_plot.collapsed(coefficients, "Show the full coefficient table")
except (ValueError, RuntimeError) as error:
    # The catchment sum is the default predictor and does not depend on this,
    # so a failure here is reported rather than allowed to stop the notebook.
    print(f"FTS returned nothing usable: {error}")
    print("\nContinuing on the catchment sum.")


## Step 2b - The second input: supraglacial lakes

Runoff is not the only thing the ice sheet does with its meltwater. Some of it
collects in lakes on the surface, and where those lakes form, how large they
grow and when they drain is a different signal from how much water the climate
model says melted.

This is the input the FTS module was built for. A catchment's runoff collapses
sensibly to one number per season; its lakes do not - lake cover is a *pattern*
across the catchment, and the shape of that pattern through a season is the
quantity of interest. So the lake field goes through exactly the same two
modules: the GRF puts it on the same 295 nodes, and the FTS compresses each
season of it into a few coefficients.

One thing the data decides for us. By September the lakes have drained and
almost the whole catchment sits at the floor of the record - in 2018 and 2020
every single cell does. A field with no variation in it has no spatial pattern
to fit, so the GRF module writes those months out as that constant instead of
fitting them. Every season then runs May to September, five months, which is
what lets the FTS see a whole melt season rather than most of one.

The lake record is shorter than the runoff record - 2014 to 2021, eight melt
seasons against twenty - because the source product does not go back further.
Anything that uses both is limited to those eight.


In [ ]:
#@title Run both modules on the lake record {display-mode: "form"}
lake_coefficients = None

# Months when the lakes have all drained have no spatial variation anywhere, so
# there is no field for the GRF to estimate. The module writes those months out
# as the constant they already are rather than dropping them, which is what
# keeps every season five months long - the length the FTS needs to see a whole
# melt season rather than most of one.
lake_geozarr_url = await api.run_grf_job(client, LAKE_DATASET, START_DATE, END_DATE,
                                         progress=set_message)
lake_ds = xr.open_dataset(lake_geozarr_url, engine="zarr")
lake = uc1.grf_fields(lake_ds)
lake_fields, lake_points = lake["fields"], lake["points"]
lake_in_basin = (uc1.basin_nodes(lake_points, basin)
                 & np.isfinite(lake_fields.to_numpy()).all(axis=0))
print(f"lake variable: {lake['variable']}")
print(f"{lake_fields.shape[0]} lake fields x {lake_fields.shape[1]} grid cells, "
      f"{lake_in_basin.sum()} cells inside the catchment outline, "
      f"{lake_fields.index.min():%b %Y} to {lake_fields.index.max():%b %Y}")

try:
    lake_fts_geozarr_url = await api.run_fts_job(client, lake_geozarr_url, curve_time_points=5,
                                                 num_pca=2, num_factors=3,
                                                 progress=set_message)
    lake_fts_ds = xr.open_dataset(lake_fts_geozarr_url, engine="zarr")
    lake_fts = uc1.fts_coefficients(lake_fts_ds)
    lake_coefficients = lake_fts["coefficients"]
    lake_coefficients.index = sorted(
        {t.year for t in lake_fields.index})[:len(lake_coefficients)]
    print(lake_fts["layout"])
    print(f"{lake_coefficients.shape[1]} coefficients describe each of the "
          f"{len(lake_coefficients)} lake seasons")
except (ValueError, RuntimeError) as error:
    # Step 3 compares the predictors that are available, so the notebook
    # carries on without this one rather than stopping.
    print(f"FTS returned nothing usable for the lake record: {error}")

if lake_in_basin.any():
    uc1_plot.plot_lake_season(lake_fields, lake_points, lake_in_basin, basin,
                              mesh_vertices, mesh_triangles)
    plt.show()


The left-hand panel is the lake field on the same mesh the runoff fields use -
that is the harmonisation step doing its job on a completely different kind of
measurement. The right-hand panel is that field reduced to one number a month,
the catchment mean, so the rise and fall of a lake season is readable. The FTS
module itself works on the whole pattern, a curve at every node. The
biggest and the smallest of the eight are drawn darker and named, so the spread
between a strong lake year and a weak one is readable.

Whether those curves help predict discharge is tested in Step 3, alongside the
runoff predictors, rather than assumed.


## Input: measured discharge

Daily discharge at the Watson gauge, 2006–2025. This is what the model is
trained against and judged on. Every year is drawn on the same axis so the
spread between seasons is visible: the largest melt seasons deliver about
three and a half times the water of the smallest, and peak four times as high.


In [ ]:
#@title Load the measured discharge record {display-mode: "form"}
gauge_geozarr_url = await api.download_dataset(client, "UC1_WATSON_DISCHARGE_GAUGE", START_DATE, END_DATE,
                                               progress=set_message)
gauge_ds = xr.open_dataset(gauge_geozarr_url, engine="zarr")
discharge = uc1.discharge_series(gauge_ds)
observed = uc1.step_matrix(discharge, STEP_DAYS)

uc1_plot.plot_discharge_record(discharge, observed)
plt.show()

print(f"{len(discharge):,} daily values, {len(observed)} melt seasons, "
      f"{len(observed.columns)} steps of {STEP_DAYS} days each")


## Step 3 - One number per season

The chain arrives at a single number describing each melt season: the catchment
total of the fitted runoff fields, the FTS coefficient derived from them, or the
FTS coefficient derived from the lake fields. All three are compared here
against the measured seasonal volume, so `PREDICTOR` is chosen on evidence
rather than by default. An FTS predictor appears only if its own step returned
something usable: if that job fails, Step 2 or Step 2b says so and the notebook
carries on with the catchment sum.

On this catchment the plain catchment sum predicts the seasonal volume better
than either FTS coefficient. That is a finding about *this* target - one integrated number at
one gauge - not a general verdict on the module: a compression designed to
preserve seasonal shape is bound to lose ground to a total when a total is what
is being predicted. What the lake coefficient adds on top of the catchment sum
is a different question, and is measured after the validation below.


In [ ]:
#@title Reduce each season to one number {display-mode: "form"}
PREDICTOR = predictor_dropdown.value

measured_total = pd.Series(
    {year: uc1.annual_total(row, STEP_DAYS) for year, row in observed.iterrows()})

candidates = {"catchment sum":
              uc1.seasonal_predictor(monthly_runoff).rename("catchment sum")}
if coefficients is not None:
    candidates["FTS factor 1"] = coefficients["factor_1"].rename("FTS factor 1")
if lake_coefficients is not None:
    candidates["lake FTS factor 1"] = (
        lake_coefficients["factor_1"].rename("lake FTS factor 1"))

uc1_plot.plot_predictor_candidates(candidates, measured_total)
plt.show()

if PREDICTOR not in candidates:
    raise ValueError(
        f"The {PREDICTOR!r} predictor is not available in this run: its FTS step returned "
        f"nothing usable (see the messages under Step 2 and Step 2b). Choose one of "
        f"{list(candidates)} as the Predictor in Setup, then run this cell again.")
predictor = candidates[PREDICTOR]
predictors = predictor.to_frame()
print(f"using {PREDICTOR!r} for everything below")


## Step 4 - Fit one relationship per week

For each step of the melt season separately, a straight line is fitted across
years between the seasonal runoff number and the discharge measured in that
step. Twenty-two weeks gives twenty-two lines.

The per-step r² printed below is worth looking at. It is *lower* than the
seasonal r² above, and that is expected: an individual week is partly weather,
which a seasonal number cannot know about. The model gets the size of the
season right and scales a characteristic shape by it.


In [ ]:
#@title Fit the weekly relationships {display-mode: "form"}
model = uc1.ProfileModel.fit(discharge, predictors, STEP_DAYS)
print(f"{len(model.starts)} steps of {model.step_days} days, fitted across "
      f"{len(model.years)} seasons ({model.years.min()}-{model.years.max()}) "
      f"on {predictors.columns[0]!r}. Median r2 within a step: "
      f"{float(pd.Series(model.r2).median()):.2f}")

fig, ax = plt.subplots(figsize=(11, 3.6))
ax.bar(model.starts, model.r2, width=STEP_DAYS * 0.8, color=OBS)
uc1_plot.month_axis(ax)
ax.set_ylabel("r$^2$ within the step")
ax.set_ylim(0, 1)
ax.set_title("How well each step of the season is predicted", loc="left")
plt.show()


## Step 5 - Validation: leave one year out

**USER INPUT** - set **Season to hold back** to any year in the record to see how the model
does on a season it was not trained on.

Each year in turn is removed, the model refitted on the rest, and that year
predicted blind. The skill printed below is across *every* held-out step of
*every* year, so it is the figure to judge it on.


In [ ]:
#@title Choose the season to hold back {display-mode: "form"}
leave_out_dropdown = widgets.Dropdown(
    options=[int(year) for year in predictors.index],
    value=2012 if 2012 in predictors.index else int(predictors.index[0]),
    description="Season to hold back:",
    style={"description_width": "initial"},
)
ipydisplay(leave_out_dropdown)

In [ ]:
#@title Validate by leaving one season out {display-mode: "form"}
LEAVE_OUT = leave_out_dropdown.value

validation = uc1.leave_one_year_out(discharge, predictors, STEP_DAYS)
skill = uc1.profile_skill(validation)
print(f"Held out, every season: r2 = {skill['r2']:.2f}, "
      f"typical error {skill['rmse_m3s']:.0f} m3/s "
      f"({skill['n_steps']} steps tested, none seen by the model that predicted it)")

held_out = uc1.ProfileModel.fit(
    discharge[discharge.index.year != LEAVE_OUT],
    predictors.drop(index=LEAVE_OUT), STEP_DAYS)
band = held_out.bands(predictors.loc[LEAVE_OUT].to_numpy(), levels=(0.68, 0.95))

uc1_plot.plot_prediction_band(band, observed.loc[LEAVE_OUT],
                              f"{LEAVE_OUT} predicted, year held out", f"{LEAVE_OUT} measured",
                              f"{LEAVE_OUT}: blind prediction against measurement")
plt.show()


## Step 5b - Using the season's own shape

Everything so far has described a melt season with a single number. The climate
model knows more than that. It also says *when* the runoff arrives, and that
within-season shape has been thrown away.

The shape model keeps it. It reads the monthly runoff curve, interpolates it
across the weekly steps, shifts it by a lag for the time water takes to reach
the gauge, and fits one straight line from that curve to measured discharge.
One relationship instead of twenty-two, and the shape of a predicted season now
comes from the forcing rather than from the average of every past season.

It is the more accurate of the two, and from here on it is the model the
notebook uses. The simple one stays for the two jobs a curve cannot do: a
climate scenario, which is one number a season by construction, and the
propagated seasonal uncertainty in Step 9.


In [ ]:
#@title Fit the shape model and compare the two {display-mode: "form"}
shape_model = uc1.ShapeModel.fit(discharge, monthly_runoff, STEP_DAYS)
shape_validation = uc1.shape_leave_one_year_out(discharge, monthly_runoff, STEP_DAYS)
shape_skill = uc1.profile_skill(shape_validation)
print(shape_model)
print(f"\nHeld out, every season: r2 = {shape_skill['r2']:.2f}, typical error "
      f"{shape_skill['rmse_m3s']:.0f} m3/s, against {skill['rmse_m3s']:.0f} m3/s "
      f"for one number a season")

# Refitted without the held-out season, as the band above is, so the two
# panels are both blind predictions and can be compared.
shape_held_out = uc1.ShapeModel.fit(
    discharge[discharge.index.year != LEAVE_OUT], monthly_runoff, STEP_DAYS)
shape_band = shape_held_out.bands(LEAVE_OUT, levels=(0.68, 0.95))

uc1_plot.plot_prediction_band(shape_band, observed.loc[LEAVE_OUT],
                              f"{LEAVE_OUT} from the season's shape", f"{LEAVE_OUT} measured",
                              f"{LEAVE_OUT}: the same season, predicted from the runoff curve")
plt.show()


### What the lakes add

The lake record covers eight melt seasons, 2014 to 2021, so this comparison is
restricted to those. The first three rows are fitted the same way as the
validation above, with every season held out of its own prediction in turn:
once on catchment runoff alone, once on the lake coefficient alone, and once on
both together. The last two ask the same question of the shape model, on the
same held-out seasons.


In [ ]:
#@title Compare runoff, lakes, and both together {display-mode: "form"}
if lake_coefficients is not None:
    runoff_only = candidates["catchment sum"]
    lake_only = lake_coefficients["factor_1"].rename("lake FTS factor 1")
    together = pd.DataFrame({"catchment runoff": runoff_only,
                             "lake pattern": lake_only}).dropna()
    lake_seasons = together.index

    rows = {}
    for name, candidate in (("runoff only", runoff_only.loc[lake_seasons]),
                            ("lakes only", lake_only.loc[lake_seasons]),
                            ("runoff + lakes", together)):
        fold_skill = uc1.profile_skill(
            uc1.leave_one_year_out(discharge, candidate, STEP_DAYS))
        rows[name] = {"held-out r2": fold_skill["r2"],
                      "held-out RMSE (m3/s)": fold_skill["rmse_m3s"]}

    # The same question of the shape model. Both rows below are fitted on the
    # same seven-of-eight folds, so they are a fair pair with each other.
    lake_discharge = discharge[discharge.index.year.isin(lake_seasons)]
    lake_monthly = monthly_runoff[monthly_runoff.index.year.isin(lake_seasons)]
    lake_extras = lake_only.loc[lake_seasons].to_frame()
    for name, extras in (("runoff curve", None),
                         ("runoff curve + lakes", lake_extras)):
        fold_skill = uc1.profile_skill(uc1.shape_leave_one_year_out(
            lake_discharge, lake_monthly, STEP_DAYS, extras=extras))
        rows[name] = {"held-out r2": fold_skill["r2"],
                      "held-out RMSE (m3/s)": fold_skill["rmse_m3s"]}

    contribution = pd.DataFrame(rows).T
    ipydisplay(contribution.round(3))

    uc1_plot.plot_lake_contribution(contribution, lake_seasons)
    plt.show()

    def _gain(better: str, base: str) -> float:
        return (1 - contribution.loc[better, "held-out RMSE (m3/s)"]
                / contribution.loc[base, "held-out RMSE (m3/s)"])

    on_a_number = _gain("runoff + lakes", "runoff only")
    on_a_curve = _gain("runoff curve + lakes", "runoff curve")
    print(f"across {len(lake_seasons)} seasons, adding the lake pattern lowers the "
          f"held-out error by {on_a_number:.0%} when the season is one number, and "
          f"by {'nothing' if on_a_curve < 0.005 else format(on_a_curve, '.0%')} "
          f"once the runoff curve is in the model")
else:
    print("The lake coefficients are not available in this run, so their "
          "contribution cannot be measured here.")


Lakes alone predict less well than runoff alone, which is the expected shape:
lake cover is a consequence of melt rather than a substitute for measuring it.
The third row is where the Earth observation contribution shows. Adding the lake
pattern to a single number per season lowers the held-out error by about an
eighth, on seasons the model had not seen, and the lake product earns that
honestly: it is derived from classified optical imagery and passes through both
modules of the chain.

The last two rows ask the same question of the shape model and answer it
differently. Once the runoff curve is in the model the lakes add nothing: 174
m³/s without them and 175 with them, on the same folds. What the lake record was
supplying is *timing*, and a model that already reads the shape of the season
has it. So the lake record earns its keep wherever the forcing cannot supply a
within-season curve, and not where it can. Which of those two comparisons is
the fair one depends on what you are driving the chain with.

The caveat either way is the record length. Eight seasons, with two predictors
and an intercept fitted on seven of them at a time, is thin, and a longer lake
record is the single thing that would most strengthen any of this.


## Step 6 - Annual totals and peak flows

The two planning numbers, for every year of the record. **Total** is the volume
delivered over the melt season. **Peak** is the largest step - at the default
weekly setting, the highest seven-day mean flow.

**USER INPUT** - set **Operating limit** to your own operating limit to count how
many steps in a year exceed it.


In [ ]:
#@title Set the operating limit {display-mode: "form"}
threshold_input = widgets.BoundedIntText(
    value=1200,
    min=0,
    max=10000,
    step=50,
    description="Operating limit (m³/s):",
    style={"description_width": "initial"},
)
ipydisplay(threshold_input)

In [ ]:
#@title Annual totals and peak flows {display-mode: "form"}
THRESHOLD_M3S = threshold_input.value

shape_profiles = shape_model.profiles(shape_model.years)
modelled = uc1.summarise(shape_profiles, STEP_DAYS)
actual = uc1.summarise(observed.loc[shape_model.years], STEP_DAYS)

table = pd.DataFrame({
    "total, measured (km3)": actual["total_km3"],
    "total, modelled (km3)": modelled["total_km3"],
    "peak, measured (m3/s)": actual["peak_m3s"],
    "peak, modelled (m3/s)": modelled["peak_m3s"],
    "peak week, measured": actual["peak_week_starting"],
    "peak week, modelled": modelled["peak_week_starting"],
    "steps above threshold, measured": uc1.steps_above(
        observed.loc[shape_model.years], THRESHOLD_M3S),
    "steps above threshold, modelled": uc1.steps_above(shape_profiles,
                                                       THRESHOLD_M3S),
})
uc1_plot.collapsed(table, "Show the season-by-season table")
print("Measured against modelled, fitted on all twenty seasons - these are "
      "in-sample figures, not the held-out skill above.")

uc1_plot.plot_totals_and_peaks(table, THRESHOLD_M3S)
plt.show()

for label, column in (("total", "total_km3"), ("peak", "peak_m3s")):
    score = np.corrcoef(modelled[column], actual[column])[0, 1] ** 2
    print(f"{label:6s} r2 = {score:.3f}")


> **Two limitations worth knowing.**
>
> *Timing.* The shape model takes its timing from the climate model's own
> season, so the peak can move: across the twenty seasons it lands between 16
> July and 14 August, against 9 July to 7 August measured. That is a real
> improvement on describing a season with one number, which could only ever put
> the peak between 9 and 24 July. It is still the weather of a particular week
> that decides the sharpest peaks, and the forcing is monthly.
>
> *Peak size.* Modelled peaks are low across the record, not only in the
> extreme years: they fall short in eighteen of twenty seasons, by about 19%
> on average.
> The reason is that each week is predicted as an average rather than as a
> particular week of weather, so the highest point of a smoothed season sits
> below the highest point of a measured one at every size. Treat every
> modelled peak as a floor rather than a best estimate, and expect the count
> of weeks above an operating threshold to be low for the same reason.


## Step 7 - Does the choice of climate model matter?

RACMO and MAR are independent regional climate models. Running the whole chain
on each and comparing the answers shows how much the prediction depends on
that choice.

There is a second, separate question: whether resolution matters. `RACMO` is
the model's 1 km product, which reaches this chain at 2 km, and `RACMO55` is its
5.5 km run, so comparing the two isolates resolution from the choice of model.
On this catchment it makes almost no
difference to the prediction, which is why `RACMO55` is the default: it covers
2006–2025 rather than stopping in 2020, and the two are indistinguishable in
held-out r² - over its own twenty seasons against the fifteen the 2 km product
reaches, so close rather than strictly comparable.


In [ ]:
#@title Compare the three runoff datasets {display-mode: "form"}
comparison, forcing_profiles = {}, {}
for name in ("RACMO55", "RACMO", "MAR"):
    # Each forcing is fitted by the deployed module in turn. The grid is the
    # same every time, but which cells carry a value is not, so each gets its
    # own mask rather than borrowing the one from Step 1.
    # The forcing chosen in Setup was already fitted in Step 1, so it is reused.
    if name == FORCING:
        other = grf
    else:
        set_message(f"Running the chain on {LABELS[name]}...")
        other_geozarr_url = await api.run_grf_job(client, DATASET[name], START_DATE, END_DATE,
                                                  progress=set_message)
        other_ds = xr.open_dataset(other_geozarr_url, engine="zarr")
        other = uc1.grf_fields(other_ds)
    other_fields = other["fields"]
    other_mask = (uc1.basin_nodes(other["points"], basin)
                  & np.isfinite(other_fields.to_numpy()).all(axis=0))
    other_monthly = uc1.catchment_mean(other_fields, other_mask)
    other_monthly = other_monthly[other_monthly.index.month.isin(uc1.MELT_MONTHS)]
    forcing_model = uc1.ShapeModel.fit(discharge, other_monthly, STEP_DAYS)
    seasonal = uc1.seasonal_predictor(other_monthly).rename(f"{LABELS[name]} runoff")
    held = uc1.profile_skill(uc1.shape_leave_one_year_out(
        discharge, other_monthly, STEP_DAYS))
    forcing_profiles[name] = forcing_model.profiles(forcing_model.years)
    in_sample = uc1.summarise(forcing_profiles[name], STEP_DAYS)
    truth = uc1.summarise(observed.loc[forcing_model.years], STEP_DAYS)
    comparison[name] = {
        "seasons": len(seasonal),
        "covers": f"{seasonal.index.min()}-{seasonal.index.max()}",
        "held-out r2": held["r2"],
        "held-out RMSE (m3/s)": held["rmse_m3s"],
        "total r2 (in sample)": np.corrcoef(in_sample["total_km3"],
                                           truth["total_km3"])[0, 1] ** 2,
        "peak r2 (in sample)": np.corrcoef(in_sample["peak_m3s"],
                                          truth["peak_m3s"])[0, 1] ** 2,
    }

uc1_plot.collapsed(pd.DataFrame({LABELS[k]: v for k, v in comparison.items()}).T,
                   "Show the forcing-by-forcing table", decimals=3)

# Compared on the seasons every forcing covers.
shared = sorted(set.intersection(*(set(p.index) for p in forcing_profiles.values())))
totals = pd.DataFrame({name: uc1.summarise(p.loc[shared], STEP_DAYS)["total_km3"]
                       for name, p in forcing_profiles.items()})
totals["measured"] = uc1.summarise(observed.loc[shared], STEP_DAYS)["total_km3"]
mean_seasons = pd.DataFrame({name: p.loc[shared].mean(axis=0)
                             for name, p in forcing_profiles.items()})
mean_seasons["measured"] = observed.loc[shared].mean(axis=0)

uc1_plot.plot_forcing_comparison(totals, mean_seasons, LABELS)
plt.show()

print(f"Different climate model (RACMO 5.5 km vs MAR 5 km):  r = "
      f"{np.corrcoef(totals['RACMO55'], totals['MAR'])[0, 1]:.3f}")
print(f"Downscaling only (RACMO 5.5 km vs RACMO 2 km):  r = "
      f"{np.corrcoef(totals['RACMO55'], totals['RACMO'])[0, 1]:.3f}")


Read this as a sensitivity test. All three agree closely on annual totals. Each
forcing is fitted against the same gauge before they are compared, so this does
not show that any of them is free of bias - a constant offset is absorbed by its
own regression. What it shows is that the three carry the same seasonal signal,
which is what makes them interchangeable here. The two resolutions of the same
model agree almost exactly - resolution barely moves the answer -
while the two different models agree slightly less well, which is the more
meaningful of the two comparisons. `RACMO55` is the default because it matches
the 1 km product's held-out r² while covering five more seasons.
On the evidence here the three are close enough that nothing in this notebook
separates them, though the held-out figures are each over that forcing's own
seasons rather than the fifteen they share.


## Step 8 - How sensitive is the river to wetter seasons?

The simple model takes over here, and for Step 9. A scenario is one number a
season by construction, so there is no within-season curve for the shape model
to read - which is the clearest statement of what each of the two is for.

The model is fitted on the record as observed, then run forward over seasons
that have not happened.

Rather than assert a particular future, the notebook reshapes the observed
distribution of seasonal runoff and asks what the model returns. Each scenario
is set by two numbers:

- **`weight`** - 0.5 centres a scenario on the observed mean, 1 on one
  standard deviation above it, 0 on one below
- **`variability`** - 0 for no spread between years, 1 for the observed spread,
  2 for twice it

So "warmer" means seasons generated around a centre shifted up from the
observed mean. They are random draws rather than measured seasons reused, and
nothing holds them inside the record, so an individual scenario season can fall
outside anything yet observed - which is also why the caveat below matters. These are illustrative
scenarios, and should be read as *what would this catchment do if seasons looked
like this*, not as a forecast.

A scenario-driven MAR or RACMO run would go through the same chain in place of
the perturbations, if one were available for this catchment and forcing.

**USER INPUT** - edit the scenarios below.

> Results are not necessarily robust at very strong weighting and high
> variability, since that pushes the forcing outside the range the model was
> trained on.


In [ ]:
#@title Choose the climate scenarios {display-mode: "form"}
DEFAULT_SCENARIOS = [
    {"name": "Warmer",        "weight": 0.8, "variability": 1.0},
    {"name": "Much warmer",   "weight": 1.0, "variability": 1.0},
    {"name": "More variable", "weight": 0.5, "variability": 1.8},
]

scenario_widgets = [
    (
        widgets.Text(value=spec["name"], description=f"Scenario {n}:",
                     style={"description_width": "initial"}),
        widgets.FloatSlider(value=spec["weight"], min=0.0, max=1.0, step=0.05,
                            description="weight:", style={"description_width": "initial"}),
        widgets.FloatSlider(value=spec["variability"], min=0.0, max=2.0, step=0.1,
                            description="variability:", style={"description_width": "initial"}),
    )
    for n, spec in enumerate(DEFAULT_SCENARIOS, start=1)
]
ipydisplay(widgets.VBox([widgets.HBox(list(row)) for row in scenario_widgets]))

In [ ]:
#@title Build the climate scenarios {display-mode: "form"}
SCENARIOS = [
    {"name": name.value, "weight": weight.value, "variability": variability.value}
    for name, weight, variability in scenario_widgets
]

runs = {"Observed record": model.profiles(predictors)}

print("Scenarios built by perturbing the observed record.\n")
for spec in SCENARIOS:
    scenario = uc1.Scenario.from_perturbation(
        spec["name"], predictor, weight=spec["weight"],
        variability=spec["variability"])
    runs[scenario.name] = scenario.run(model)
    print(scenario)

summary = uc1.compare(runs, STEP_DAYS)
uc1_plot.collapsed(summary, "Show the scenario table")


### The scenarios, drawn

Left: every season of every scenario, so the spread is visible. Right: the mean
season under each, which is the comparison to read.


In [ ]:
#@title Draw the scenarios {display-mode: "form"}
uc1_plot.plot_scenarios(runs)
plt.show()

### What that means for planning

Each scenario is summarised by the two planning numbers used above: the volume
the season delivers, and how large its peak gets.


In [ ]:
#@title Summarise what the scenarios mean for planning {display-mode: "form"}
uc1_plot.plot_scenario_summary(summary)
plt.show()


## Step 9 - Uncertainty, propagated

Two of the reasons the prediction could be wrong can be added together here,
rather than confused with one another. This step is the simple model again: it
propagates a seasonal number and its spread, which is the form the module's
uncertainty arrives in.

**Fitting uncertainty.** Each weekly relationship has scatter around it, and
that scatter is what the band in Step 5 shows, widened where a season sits far
from the middle of the record. The prediction at the centre of that band is held
out; its width is the fit's own residual spread rather than a tally of held-out
misses. The held-out error is the separate figure printed in Step 5.

**Harmonisation uncertainty.** The GRF module does not return a single field -
it returns a *posterior*, with a credible interval at every mesh node. Averaging
those to one catchment number carries a spread, and that spread should widen the
final band.

`uc1_hydro_helpers.propagate` combines both by Monte Carlo: the seasonal number is
resampled from its own posterior spread, and a residual is drawn for every step
of every draw.

One limit. Averaging node posteriors is not averaging independent numbers - a
fitted field is smooth, so neighbouring nodes err together. Where the module
publishes the spread of the spatial mean itself, which it does here, that
joint-posterior number is the one used. `catchment_sd` falls back to collapsing
the per-node spreads, and takes the conservative limit when it does.

What is not in this band is the choice of forcing. Step 7 measures that
separately, and it is not added in here.


In [ ]:
#@title Load the harmonisation spread {display-mode: "form"}
seasonal_spread = None
sd_fields = grf["uncertainty"]

if PREDICTOR != "catchment sum":
    print(f"The harmonisation spread is measured on catchment runoff, so it does not "
          f"apply to {PREDICTOR!r}. The sweep below shows what such a term would do.")
elif sd_fields is None:
    print("The GRF output carries no posterior spread, so the harmonisation term")
    print("cannot be computed here.")
    print("")
    print("The sweep below shows what it would do to the band.")
else:
    # Averaging node posteriors is not averaging independent numbers, so where
    # the module publishes the spread of the spatial mean itself - taken from
    # the joint posterior - that is the one to use. Collapsing the per-cell
    # spreads is the fallback, and it is the conservative limit.
    if grf["uncertainty_mean"] is not None:
        per_snapshot = grf["uncertainty_mean"].dropna()
        source_note = "the module's own uncertainty_mean, from the joint posterior"
    else:
        per_snapshot = uc1.catchment_sd(sd_fields.loc[:, in_basin])
        source_note = "collapsed from the per-cell spreads"
    factor = uc1.multiplicative_factor(per_snapshot)
    print(f"GRF posterior spread: {source_note}")
    print(f"typical value {per_snapshot.mean():.3f} in log10 units "
          f"-> a factor of {factor:.2f} on the runoff itself")

    # The spread is in log10 units, so it scales the back-transformed monthly
    # means rather than adding to them.
    seasonal_spread = uc1.seasonal_sd(per_snapshot,
                                      [t.year for t in per_snapshot.index],
                                      monthly_values=monthly_runoff, log10=True)
    seasonal_spread = seasonal_spread.reindex(predictors.index).dropna()
    print(f"\nspread on the seasonal number, {len(seasonal_spread)} seasons: "
          f"{seasonal_spread.mean():.1f} mm water equivalent "
          f"({seasonal_spread.mean() / predictor.mean():.0%} of its value)")

    if factor > 1.5:
        print(f"\nA factor of {factor:.1f} on catchment runoff is wider than a fitted")
        print("field should be, which would mean the reported spread is not a")
        print("posterior standard deviation of the field. The number below should")
        print("not be quoted until that is resolved.")


In [ ]:
#@title Propagate the uncertainty through one season {display-mode: "form"}
example_year = LEAVE_OUT if LEAVE_OUT in predictors.index else predictors.index[-1]
example_value = predictors.loc[example_year].to_numpy()

if seasonal_spread is not None and example_year in seasonal_spread.index:
    spread = float(seasonal_spread.loc[example_year])
    ipydisplay(uc1.uncertainty_budget(model, example_value, spread).round(3))

    fitting = uc1.propagate(model, example_value, 0.0)
    total = uc1.propagate(model, example_value, spread)

    uc1_plot.plot_uncertainty_band(fitting, total, observed.loc[example_year], example_year)
    plt.show()
else:
    # A sensitivity sweep, not a result: it says how much the band *would*
    # widen for a given harmonisation spread, so the term can be judged before
    # it is available.
    fractions = [0.0, 0.02, 0.05, 0.10, 0.20]
    widths = []
    for fraction in fractions:
        swept = uc1.propagate(model, example_value,
                              fraction * float(predictor.mean()))
        widths.append(float((swept["upper"] - swept["lower"]).mean()))

    uc1_plot.plot_spread_sensitivity(fractions, widths)
    plt.show()

    print("Illustrative only - these are assumed spreads, not measured ones.")
    print(f"Fitting alone gives a band {widths[0]:.0f} m3/s wide. A harmonisation")
    print(f"spread of 5% would widen it to {widths[2]:.0f}, and 20% to {widths[4]:.0f}.")


### Is the band the right width?

The check is how much of the measured record falls inside it. A band claiming
68% should contain about 68% of measured values, and one claiming 95% about 95%.
Much more and it is too wide to be useful; much less and it is overconfident.

The model used here is the one fitted on all twenty seasons, so this tests the
width of the band against the record that set it, not against seasons it never
saw. Step 5 is where the held-out figure is.

Two coverages are offered because they answer different questions. **68%** is
the natural one-sigma summary and is what you would quote for a typical year.
**95%** is what infrastructure design asks for - a spillway is not sized on a
typical year - and it is the one to use when the answer feeds a decision about
capacity or resilience.

Set **Interval to check** below to whichever you want checked.


In [ ]:
#@title Choose the interval to check {display-mode: "form"}
level_dropdown = widgets.Dropdown(
    options=[("68% (one sigma)", 0.68), ("95% (design interval)", 0.95)],
    value=0.95,
    description="Interval to check:",
    style={"description_width": "initial"},
)
ipydisplay(level_dropdown)

In [ ]:
#@title Test whether the band is the right width {display-mode: "form"}
LEVEL = level_dropdown.value

inside = {}
for year in predictors.index:
    if year not in observed.index:
        continue
    spread = (float(seasonal_spread.loc[year])
              if seasonal_spread is not None and year in seasonal_spread.index
              else 0.0)
    year_band = uc1.propagate(model, predictors.loc[year].to_numpy(), spread,
                              level=LEVEL)
    row = observed.loc[year]
    inside[year] = ((row >= year_band["lower"]) & (row <= year_band["upper"])).mean()
inside = pd.Series(inside)

print(f"{inside.mean():.0%} of measured steps fall inside the {LEVEL:.0%} band")
print(f"(about {LEVEL:.0%} would be expected if it were exactly right)")

fig, ax = plt.subplots(figsize=(11, 3.4))
ax.bar(inside.index, inside.to_numpy(), color=OBS, width=0.7)
ax.axhline(LEVEL, color=MODEL, ls="--", lw=1.8)
ax.set_ylabel("fraction of steps inside band")
ax.set_ylim(0, 1.05)
ax.set_title("Coverage of the prediction interval, by season", loc="left")
plt.show()


## Is the model worth having?

Everything above assumes the chain earns its place. This section checks that,
by comparing it against the simplest thing anyone could do instead.

**Assume every season is average.** Predict each season as the mean of all the
others. No climate model, no gauge relationship, nothing but the long-term
average shape of a melt season. This is the null model any prediction has to
beat to be worth making.

**Assume runoff is discharge.** Take the climate model's runoff over the
catchment, convert it to a flow, and call that the river. No gauge record, no
fitting. It is what the climate model alone can tell you.

**The model.** One number per season, twenty-two weekly relationships, every
year predicted with itself held out - what the rest of this notebook does.

**The model, using the shape of the season.** The same idea, but instead of
compressing the season to one total it uses the monthly runoff as a curve,
shifted by a lag for the time water takes to reach the gauge. The lag and the
fit are both chosen without the held-out year.

All four are scored the same way: root-mean-square error against measured
discharge, across every week of every season.


In [ ]:
#@title Compare against simpler baselines {display-mode: "form"}
# The null model: every season predicted as the average of all the others.
# Scored on the same seasons as the other three so the rows are comparable.
scored_seasons = (observed.index.intersection(predictors.index)
                  .intersection(monthly_runoff.index.year.unique()))
mean_season = pd.DataFrame(index=scored_seasons, columns=observed.columns, dtype=float)
for year in scored_seasons:
    mean_season.loc[year] = observed.drop(index=year).mean(axis=0).to_numpy()
climatology = pd.concat({"observed": observed.loc[scored_seasons],
                         "predicted": mean_season}, axis=1)

baselines = {
    "Mean season (no model at all)": climatology,
    "Runoff assumed to be discharge":
        uc1.runoff_as_discharge(monthly_runoff, discharge, STEP_DAYS).loc[scored_seasons],
    "Current model": uc1.leave_one_year_out(discharge, predictors.loc[scored_seasons], STEP_DAYS),
    # the same held-out fits as Step 5b, so they are reused rather than refitted
    "Model using the season's shape": shape_validation.loc[scored_seasons],
}

scores = pd.DataFrame({name: uc1.profile_skill(v) for name, v in baselines.items()}).T
ipydisplay(scores.round(3))

# By how much, and is it a constant offset? Compare the naive prediction with
# the gauge over the whole season and over its first and last thirds.
naive = baselines["Runoff assumed to be discharge"]
pred, obs = naive["predicted"], naive["observed"]
both = pred.notna() & obs.notna()
pred, obs = pred.where(both), obs.where(both)
third = max(len(obs.columns) // 3, 1)


def _ratio(columns: slice) -> float:
    return (pred.iloc[:, columns].sum().sum()
            / obs.iloc[:, columns].sum().sum())


print(f"Taking {LABELS[FORCING]} runoff straight as discharge, shifted by one "
      f"week for travel time, gets the season total to within "
      f"x{_ratio(slice(None)):.2f} of the gauge. The error is not a constant "
      f"offset: x{_ratio(slice(0, third)):.2f} across the first third of the "
      f"melt season against x{_ratio(slice(-third, None)):.2f} across the last.")

uc1_plot.plot_baseline_errors(baselines)
plt.show()

The first two lines are the point. With the default settings, assuming every season is average gives 297 m3/s of
weekly error; taking the climate model's runoff as the river gives 245.
Over a whole season that naive version is within about 7% of the right volume -
the factor printed above is x1.07 - but it puts the water in the wrong weeks,
running about 55% high early and 26% high at the end. The climate model on its own, in
other words, barely improves on knowing nothing about the particular year, and
the part of its signal that matters only emerges once a relationship to the
gauge is fitted. That fitted model reaches 217, a 27% cut against the null, and
using the within-season shape reaches 185 by improving the timing, because a
seasonal total cannot say when the water arrives.

The shape model is the more accurate of the two, and it is where this work
goes next. The simpler one is the default here for a reason that matters to
a digital twin: it needs only a single number per season, so any source that
can produce one - a different climate model, a coarser satellite product, a
scenario run - drives it without retuning. The comparison above is the price
of that interchangeability, about 32 m3/s of weekly error.

All four struggle with the same seasons, which is worth noticing: none of
them sees the weather that makes an individual week extreme.


## What this shows

For one catchment and twenty melt seasons, the chain produces:

- a fitted field for every month of every season, all on the same 295-node
  mesh, whatever the source it came from
- one number describing each season, taken from those fields
- weekly discharge through the season, with a 68% and a 95% band around it
- the season's total and its peak, and the count of weeks above an operating
  threshold you choose
- the same four, run forward under climate scenarios

And it shows the following.

**The shape of a melt season is worth more than its size.** A single figure from
a climate model, put through twenty-two weekly relationships, reproduces seasons
the model had never seen at r² = 0.80, with a typical error of 217 m³/s across
440 held-out weeks. Reading the shape of the model's own season instead, through
one relationship rather than twenty-two, reaches r² = 0.86 and 185 m³/s across
the same held-out weeks, and lets the peak fall where the forcing puts it. Both
are here because they take different inputs: the shape model needs a monthly
curve, the simpler one needs a number, and only a number can drive a scenario.

**The answer barely depends on which climate model you use.** RACMO and MAR
agree closely on seasonal totals, and the 5.5 km and 2 km versions of RACMO
are indistinguishable. The prediction is a property of this catchment and its
gauge rather than of one model's biases - which is what makes the chain worth
running on whatever forcing is available.

**Peak size is usable for planning; peak timing is not, and peaks run low.**
Modelled peaks fall short in eighteen of twenty seasons, by about 19% on
average, because each week is predicted as an average rather than as a
particular week of weather. Treat a modelled peak as a floor, and expect counts
of weeks above an operating threshold to be low for the same reason. When the
water arrives needs a model that sees weather, not a seasonal total.

**The satellite lake record earns its place, and what it adds is timing.**
Supraglacial lake cover, mapped from optical imagery, goes through the same two
modules as the runoff and then into the same prediction. On its own it predicts
less well than runoff, as it should - lakes are a consequence of melt rather
than a measurement of it. Added to runoff it lowers the held-out weekly error by
13%, from 235 to 205 m³/s across the eight seasons the lake record covers. That
is the Earth observation contribution to this use case, measured rather than
asserted.

What it is adding is when the water arrives, not how much of it there is. The
shape model takes that timing from the runoff curve directly, and on the same
eight seasons it reaches 174 m³/s without the lake record and 175 with it. The two inputs carry the same information, so the lake record earns
its keep wherever the forcing cannot supply a within-season curve - a coarser
product, a seasonal total, a scenario run - and not where it can. Eight seasons
is a thin basis for any of it, and a longer lake record is still the single
thing that would most strengthen the result.

**Some things were tested and set aside.** CryoSat-2 seasonal elevation change
(ESA) was harmonised onto the same mesh in earlier work on this use case and
tracks discharge, but adds nothing once RACMO runoff is in the model, and it is
not used here. Compressing the *runoff* into
functional time series coefficients also predicts less well than the plain
catchment total - the compression is built to preserve the shape of a season,
which is what makes it valuable for lakes and redundant for runoff. The within-season shape model, which carries
Steps 5b to 7 here, lowers the weekly error from 217 to 185 m³/s. The simpler
model is kept rather than retired because it asks only for a single number per
season, so anything able to produce one drives it without retuning - a coarser
satellite product, a seasonal total, a scenario run - and that
interchangeability is what a digital twin needs most.

**For a planner,** that means the volume a melt season delivers, and roughly how
large its peak will be, can be estimated from climate-model output before the
season starts - with a 95% band that holds. Sizing capacity on the peak needs a
margin above what this reports, and scheduling around the peak needs something
this model does not provide.

Things worth trying:

- change **Time step** to `14` or `1` and watch the skill change
- switch **Forcing dataset** between `RACMO55`, `RACMO` and `MAR`
- push `weight` and `variability` to the edges of their ranges and see where
  the model stops being believable
- set **Operating limit** to a limit that matters to you

## Credits and data sources

**The method.** This chain follows the demonstrator David Parkes built for this
use case - one number describing a melt season, and a separate relationship
fitted for each week of it. The functional time series step implements a method
Israel Martinez Hernandez is an author of, first through his R implementation,
`model.coeff.R`, and now through a Python port of it.

**The data.**

- *Watson River discharge, 2006–2025* - measured by PROMICE and GIOS, and
  distributed by the Geological Survey of Denmark and Greenland (GEUS).
- *RACMO runoff* - Institute for Marine and Atmospheric Research, Utrecht
  (IMAU). Used here at 5.5 km, and as the 1 km product sampled at 2 km.
- *MAR runoff at 5 km* - University of Liège.
- *Supraglacial lake cover, 2014–2021*, from classified optical imagery -
  4D Greenland.

The harmonisation onto the mesh, and the modules this notebook runs, are
produced at Lancaster University and deployed in the DTC-IS environment.

**The software.** The spatial fits run in R-INLA, which is installed separately
under its own licence. The functional time series step runs in NumPy, ported
from the `hdftsa` package for R.

**Funding.** The Digital Twin Component for Ice Sheets is funded by the European
Space Agency.


We welcome your feedback, questions, and suggestions for new features. You can
reach the DTC-IS team at
[support@dtc-ice-sheets.org](mailto:support@dtc-ice-sheets.org).
